# 03 - Baseline Modeling

Stage 3 objective: build a reproducible train/validation workflow, train baseline models, compare validation metrics, and perform initial error analysis.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

def _find_project_root() -> Path:
    candidates = []
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])

    projects_dir = Path.home() / 'PycharmProjects'
    if projects_dir.exists():
        candidates.extend([path for path in projects_dir.iterdir() if path.is_dir()])

    for candidate in candidates:
        if (candidate / 'src' / 'config.py').is_file() and (candidate / 'data' / 'processed' / 'california_housing_clean.csv').is_file():
            return candidate

    raise RuntimeError('Could not locate project root with src/config.py and processed dataset')

PROJECT_ROOT = _find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    BASELINE_RESULTS_FILE,
    PROCESSED_DATA_FILE,
    RANDOM_STATE,
    SPLIT_METADATA_FILE,
    TARGET_COLUMN,
    TEST_SIZE,
    VALIDATION_SIZE,
)
from src.data import load_processed_dataset, save_report_dataframe
from src.split import create_train_val_test_split, build_split_metadata, save_split_metadata
from src.modeling import build_baseline_pipelines
from src.evaluate import (
    absolute_error_by_target_quantile,
    build_residual_frame,
    evaluate_baseline_models,
    residual_summary,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)
plt.style.use('ggplot')

## 1. Load Cleaned Data

In [ ]:
df = load_processed_dataset(PROCESSED_DATA_FILE)
print(f'Loaded cleaned data from: {PROCESSED_DATA_FILE.resolve()}')
print(f'Dataset shape: {df.shape}')
df.head()

**Conclusion:** Stage 2 cleaned data is available locally and ready for reproducible modeling experiments.

## 2. Reproducible Train/Validation/Test Setup

In [ ]:
split = create_train_val_test_split(
    df,
    target_column=TARGET_COLUMN,
    test_size=TEST_SIZE,
    val_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

split_metadata = build_split_metadata(
    split,
    random_state=RANDOM_STATE,
    test_size=TEST_SIZE,
    val_size=VALIDATION_SIZE,
)
metadata_path = save_split_metadata(split_metadata, SPLIT_METADATA_FILE)

print(f'Split metadata saved to: {metadata_path.resolve()}')
pd.Series(split_metadata, name='value')

**Conclusion:** The split is reproducible because all split operations use a fixed `random_state`. 
A held-out test subset is created now and kept untouched during baseline model comparison.

## 3. Baseline Preprocessing and Model Set

In [ ]:
pipelines = build_baseline_pipelines(random_state=RANDOM_STATE)
print('Baseline models:')
for name in pipelines:
    print('-', name)

**Preprocessing design:** feature engineering + numeric imputation + optional scaling are integrated into sklearn pipelines, 
so all fitted transforms are learned only on training data (leakage-safe).

## 4. Train Baselines and Evaluate on Validation

In [ ]:
evaluation = evaluate_baseline_models(
    model_pipelines=pipelines,
    X_train=split.X_train,
    y_train=split.y_train,
    X_val=split.X_val,
    y_val=split.y_val,
)

results_df = evaluation.results.copy()
results_path = save_report_dataframe(results_df, BASELINE_RESULTS_FILE)

print(f'Baseline results saved to: {results_path.resolve()}')
results_df

In [ ]:
ax = results_df.set_index('Model')['RMSE_val'].sort_values().plot(kind='bar', figsize=(10, 4), color='steelblue')
ax.set_title('Validation RMSE by Baseline Model')
ax.set_ylabel('RMSE')
ax.set_xlabel('Model')
plt.tight_layout()
plt.show()

**Conclusion:** Validation metrics provide a clean first ranking of baseline models without using the test set for selection.

## 5. Initial Error Analysis

In [ ]:
top_models = results_df['Model'].head(2).tolist()
error_rows = []

for model_name in top_models:
    residual_df_model = build_residual_frame(split.y_val, evaluation.validation_predictions[model_name])
    summary = residual_summary(residual_df_model).to_dict()
    summary['Model'] = model_name
    error_rows.append(summary)

error_summary_df = pd.DataFrame(error_rows).set_index('Model')
error_summary_df

In [ ]:
best_model_name = results_df.iloc[0]['Model']
best_pred = evaluation.validation_predictions[best_model_name]
best_residual_df = build_residual_frame(split.y_val, best_pred)

print(f'Best model on validation RMSE: {best_model_name}')
best_residual_summary = residual_summary(best_residual_df)
best_residual_summary

In [ ]:
error_by_quantile = absolute_error_by_target_quantile(best_residual_df, bins=5)
error_by_quantile

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(best_residual_df['actual'], best_residual_df['predicted'], alpha=0.25, s=12)
min_val = min(best_residual_df['actual'].min(), best_residual_df['predicted'].min())
max_val = max(best_residual_df['actual'].max(), best_residual_df['predicted'].max())
axes[0].plot([min_val, max_val], [min_val, max_val], color='black', linestyle='--', linewidth=1)
axes[0].set_title(f'Predicted vs Actual ({best_model_name})')
axes[0].set_xlabel('Actual')
axes[0].set_ylabel('Predicted')

axes[1].scatter(best_residual_df['predicted'], best_residual_df['residual'], alpha=0.25, s=12, color='darkorange')
axes[1].axhline(0, color='black', linestyle='--', linewidth=1)
axes[1].set_title(f'Residuals vs Predicted ({best_model_name})')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

## Stage 3 Conclusions

- Reproducible train/validation/test split is established with fixed random seed.
- Baseline preprocessing is unified through sklearn pipelines to avoid leakage.
- Baseline models are compared using validation RMSE, MAE, and R².
- Error diagnostics (residual stats, residual plot, error by target quantile) provide practical guidance for Stage 4 tuning.